# SearchQA -- Inference, Evaluation & Contrastive Training

This notebook runs the full HalluLens pipeline on SearchQA in **closed-book mode**
(search snippets are stripped -- the model relies on parametric memory).

**Dataset:** [SearchQA](https://huggingface.co/datasets/kyunghyuncho/search_qa) -- Jeopardy-style QA
- `train` split: ~99,820 questions (default for training the contrastive model)
- `validation` split: ~13,893 questions
- `test` split: ~27,247 questions

**Why SearchQA?**
Jeopardy-style factual QA with a distinctive answer distribution (proper nouns, dates,
short factoids). Broadens the hallucination-detection evaluation to yet another question
style beyond multi-hop (HotpotQA) and open-domain (TriviaQA / NQ).

**Steps:**
1. Inference -- generate model responses with activation logging
2. Evaluation -- substring match against gold answer; write `eval_results.json`
3. Inspection -- per-sample view

**Prerequisite:** A running vLLM server (`COMPUTE_CONTEXT=REMOTE_GPU` in `.env`).

## 1 -- Configuration

In [ ]:
import os, sys, json
from pathlib import Path

# -- Repo root on path --
repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
os.chdir(repo_root)

# -- Model --
MODEL = "meta-llama/Llama-3.1-8B-Instruct"
model_name = MODEL.split("/")[-1]

# -- Dataset split and sample count --
SPLIT = "train"     # "train" | "validation" | "test"
N     = None         # None = entire split

# -- Storage paths --
base_dir          = Path("shared/searchqa")
activations_path  = str(base_dir / "activations.zarr")
log_file          = str(base_dir / "server.log")

output_dir        = "output"
task_dir          = Path(output_dir) / "searchqa" / model_name
generations_file  = str(task_dir / "generation.jsonl")
eval_results_file     = str(task_dir / "eval_results.json")
eval_results_llm_file = str(task_dir / "eval_results_llm.json")

# -- Activation logging --
LOGGER_TYPE = "zarr"   # zarr (preferred) | lmdb | json

# -- Inference settings --
MAX_TOKENS  = 64
TEMPERATURE = 0.0

# -- Batched inference (ModelAdapter path) --
BATCH_SIZE = 8

# -- LLM evaluator --
LLM_EVALUATOR = None   # None -> use class default

base_dir.mkdir(parents=True, exist_ok=True)
task_dir.mkdir(parents=True, exist_ok=True)

print(f"Model             : {MODEL}")
print(f"Split             : {SPLIT}  (N={N or 'all'})")
print(f"Batch size        : {BATCH_SIZE or 'disabled (sequential server path)'}")
print(f"Generations file  : {generations_file}")
print(f"Eval (substring)  : {eval_results_file}")
print(f"Eval (LLM judge)  : {eval_results_llm_file}")
print(f"Activations       : {activations_path}")
print(f"Logger type       : {LOGGER_TYPE}")

## 2 -- Inference

Generates model responses for SearchQA questions in closed-book mode and logs activations.
Resume-safe: questions already in `generation.jsonl` are skipped.

When `BATCH_SIZE` is set, inference uses `HFTransformersAdapter.infer_batch()` directly --
multiple prompts are packed into a single `model.generate()` call for higher GPU throughput.

In [ ]:
from scripts.run_with_server import run_experiment

run_experiment(
    step="inference",
    task="searchqa",
    model=MODEL,
    split=SPLIT,
    N=N,
    logger_type=LOGGER_TYPE,
    activations_path=activations_path,
    log_file=log_file,
    output_dir=output_dir,
    generations_file_path=generations_file,
    max_inference_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    resume=True,
    batch_size=BATCH_SIZE,
)

## 3 -- Evaluation

Two evaluation methods:

- **Substring match** (`eval`) -- fast, deterministic. Correct if the gold answer appears verbatim in the response.
- **LLM judge** (`eval_llm`) -- uses an LLM to assess semantic correctness. Handles paraphrased answers.

In [ ]:
from scripts.run_with_server import run_experiment

# -- 3a: Substring match eval --
run_experiment(
    step="eval",
    task="searchqa",
    model=MODEL,
    output_dir=output_dir,
    generations_file_path=generations_file,
    eval_results_path=eval_results_file,
)

# -- 3b: LLM-judge eval --
run_experiment(
    step="eval_llm",
    task="searchqa",
    model=MODEL,
    output_dir=output_dir,
    generations_file_path=generations_file,
    eval_results_path=eval_results_llm_file,
    llm_evaluator=LLM_EVALUATOR,
    resume=True,
)

## 4 -- Results Inspection

In [ ]:
import pandas as pd

with open(eval_results_file) as f:
    substr_res = json.load(f)

n = substr_res["total_count"]
substr_correct_count = substr_res["accurate_count"]

print(f"Model          : {model_name}")
print(f"Split          : {SPLIT}  (N={n})")
print()
print(f"{'Metric':<30} {'Substring':>12}")
print("-" * 43)
print(f"{'Correct count':<30} {substr_correct_count:>12}")
print(f"{'Correct rate':<30} {substr_res['correct_rate']:>12.1%}")
print(f"{'Hallucination rate':<30} {substr_res['halu_Rate']:>12.1%}")
print("-" * 43)

In [ ]:
# Per-sample inspection
raw_df = pd.read_json(task_dir / "raw_eval_res.jsonl", lines=True)
print(f"Loaded {len(raw_df)} samples")
raw_df.head()

In [ ]:
# Show some hallucinated examples
halu_df = raw_df[raw_df["is_hallucination"]].reset_index(drop=True)
print(f"Hallucinated samples: {len(halu_df)}")
print()
for i, row in halu_df.head(6).iterrows():
    print(f"  Q  : {row['question']}")
    print(f"  Gen: {row['generation']}")
    print(f"  Ans: {row['answer']}")
    print()